# Degradación del registro de corte — exploración

Modelo del procesamiento por Dicer de un dsRNA quimérico multi-blanco.
Ver el README para la derivación completa.

El único parámetro libre es `eps`: la fracción de cortes que no mide 21 nt.


In [ ]:
import sys, pathlib
sys.path.insert(0, str(pathlib.Path.cwd().parent / "src"))

import numpy as np
import matplotlib.pyplot as plt

from fase import (
    rendimiento_un_extremo,
    rendimiento_dos_extremos,
    longitud_util,
    tasa_decaimiento,
    p_fase_cero,
)

## 1. Rendimiento por posición

Cada curva es un escenario distinto de `eps`. El eje y es la probabilidad de
que la unidad ubicada en esa posición salga cortada en fase.

In [ ]:
N = 15
eps_vals = [0.02, 0.05, 0.10, 0.20]
colores = ["#1D9E75", "#378ADD", "#BA7517", "#D85A30"]
pos = np.arange(1, N + 1)

fig, axes = plt.subplots(1, 2, figsize=(12, 4.5), sharey=True)

for e, c in zip(eps_vals, colores):
    axes[0].plot(pos, rendimiento_un_extremo(N, e), "o-", color=c, ms=4, label=f"eps = {e:.0%}")
    axes[1].plot(pos, rendimiento_dos_extremos(N, e), "o-", color=c, ms=4, label=f"eps = {e:.0%}")

axes[0].set_title("Corte desde un extremo")
axes[1].set_title("Corte desde los dos extremos")

for ax in axes:
    ax.axhline(1/21, ls=":", color="#888780", lw=1)
    ax.set_xlabel("Posición de la unidad")
    ax.set_xticks(pos)
    ax.set_ylim(0, 1)
    ax.grid(alpha=0.25, lw=0.6)
    ax.legend(frameon=False, fontsize=9)

axes[0].set_ylabel("P(la unidad sale en fase)")
plt.tight_layout()
plt.show()

La línea punteada es el piso `1/21`: aun con la fase completamente perdida,
una de cada 21 unidades cae en registro por azar.

## 2. Cuántas unidades conviene concatenar

`longitud_util` devuelve la primera posición cuyo rendimiento cae por debajo de
la mitad del de la primera unidad.

In [ ]:
print(f"{'eps':>6}  {'pos. útil':>10}  {'rend. medio':>12}  {'última/primera':>15}")
for e in [0.01, 0.02, 0.05, 0.10, 0.15, 0.20, 0.30]:
    w = rendimiento_un_extremo(N, e)
    print(f"{e:6.0%}  {longitud_util(e):>10}  {w.mean():12.3f}  {w[-1]/w[0]:15.3f}")

## 3. Un extremo vs. dos extremos

El promedio es el mismo; lo que cambia es la peor posición.

In [ ]:
for e in eps_vals:
    w1 = rendimiento_un_extremo(N, e)
    w2 = rendimiento_dos_extremos(N, e)
    print(f"eps = {e:.0%}   medio = {w1.mean():.3f}   "
          f"peor con 1 extremo = {w1.min():.3f}   peor con 2 = {w2.min():.3f}")

## 4. Sensibilidad

Mové `eps_custom`, `N_custom` y `p22_custom` para explorar otros escenarios.
`p22` es la proporción de 22 nt dentro de la fracción fuera de registro.

In [ ]:
eps_custom = 0.08
N_custom   = 15
p22_custom = 0.5

w = rendimiento_dos_extremos(N_custom, eps_custom, p22_custom)

plt.figure(figsize=(7, 4))
plt.plot(np.arange(1, N_custom + 1), w, "o-", color="#534AB7", ms=5)
plt.axhline(1/21, ls=":", color="#888780", lw=1)
plt.xlabel("Posición de la unidad")
plt.ylabel("P(la unidad sale en fase)")
plt.title(f"eps = {eps_custom:.0%},  N = {N_custom},  p22 = {p22_custom}")
plt.xticks(np.arange(1, N_custom + 1))
plt.ylim(0, 1)
plt.grid(alpha=0.25, lw=0.6)
plt.show()

print(f"rendimiento medio: {w.mean():.3f}")
print(f"peor posición:     {w.min():.3f}  (posición {int(w.argmin()) + 1})")
print(f"posición útil:     {longitud_util(eps_custom, p22_custom)}")

## 5. Validación contra Monte Carlo

El cálculo por Fourier es exacto. La simulación es el control.

In [ ]:
rng = np.random.default_rng(0)
e = 0.10
M = 21

sim = rng.choice([0, 1, 3], size=(200_000, N), p=[1 - e, e/2, e/2])
fase = np.cumsum(sim, axis=1) % M
emp = [(sim[:, 0] == 0).mean()] + [
    ((fase[:, k-1] == 0) & (sim[:, k] == 0)).mean() for k in range(1, N)
]

exacto = rendimiento_un_extremo(N, e)
print(f"discrepancia máxima: {np.max(np.abs(np.array(emp) - exacto)):.5f}")
print(f"P(Phi_200 = 0) = {p_fase_cero(200, 0.20):.4f}   (esperado 1/21 = {1/21:.4f})")